# 02 — Derive the CLI

Generate a Click interface from the CWL tool definitions and trace the resulting options to implementation callbacks.

Run this lab in the **dev-platform** workspace on Minikube or Kubernetes, using the `eoepca/pde-code-server:amd64` image. Select its preinstalled **Bash** kernel and use **Shift+Enter**, reading each explanation before executing its cell. No terminal or kernel installation is needed. Stop at an unexpected error and fix it before continuing. The course environment is provisioned by workspace startup.

The kernel keeps environment variables and the working directory between cells. The first cell finds the course checkout and activates its locked environment. Command groups run in a subshell with `set -euo pipefail` so a failing command stops that cell without killing the kernel. Scratch results stay in `.notebook-runs/02-derive-the-cli/`; rerunning a cell replaces only its lab outputs.

[Lesson](../modules/02-derive-the-cli/lesson.md) · [Exercise](../modules/02-derive-the-cli/exercise.md)


## Prepare the notebook workspace

Run this cell after every kernel restart. It clears inherited `CWLTOOL_OPTIONS` so a Coder workspace configured for Podman can still run the explicitly local exercises. It does not change your cluster or publish anything.


In [ ]:
lab_start_dir="$PWD"
while [[ ! -f reference/waterbodies.cwl || ! -f pyproject.toml ]]; do
    if [[ "$PWD" == / ]]; then
        cd "$lab_start_dir"
        break
    fi
    cd ..
done
export WORKSPACE="$PWD"
if [[ -x "$WORKSPACE/.venv/bin/python" && -f "$WORKSPACE/reference/waterbodies.cwl" ]]; then
    source "$WORKSPACE/.venv/bin/activate"
    export PYTHONPATH="$WORKSPACE${PYTHONPATH:+:$PYTHONPATH}"
    export CWLTOOL_OPTIONS=
    export MPLBACKEND=Agg
    export LAB_DIR="$WORKSPACE/.notebook-runs/02-derive-the-cli"
    mkdir -p "$LAB_DIR"
    cd "$LAB_DIR"
    python --version
    waterbodies --help
else
    printf 'Open this notebook inside the course checkout. Ask the instructor to check workspace environment provisioning.\n' >&2
    false
fi


## Generate into scratch space

The generator reads all four tools from the packed graph. Generate a separate copy so inspecting the result leaves the committed generated CLI intact. Change the contract before regenerating if an interface needs to change.


In [ ]:
(
set -euo pipefail
mkdir -p generated
transpiler-mate cwl2click --bundle --output generated "$WORKSPACE/reference/waterbodies.cwl"
find generated -maxdepth 2 -type f
)


## Read help and trace a callback

Click parses required options, checks file paths and calls the implementation. Compare `--input-item` with the crop `Directory` input and locate its callback import in the generated Python.


In [ ]:
(
set -euo pipefail
waterbodies --help
for command in crop norm-diff otsu stac; do
    waterbodies "$command" --help
done
python - <<'PYTHON'
from pathlib import Path
for path in Path('generated').rglob('*.py'):
    print('Generated file:', path)
    for line in path.read_text().splitlines():
        if 'callback' in line or 'crop_impl' in line or 'input-item' in line:
            print(line)
PYTHON
)


## Predict a missing-option failure

Calling crop without inputs must fail during argument parsing. The `if` explicitly checks that expected failure; unexpected success fails the lab.


In [ ]:
(
set -euo pipefail
if waterbodies crop > missing-options.log 2>&1; then
    printf 'Unexpected success: required options were not enforced.\n' >&2
    exit 1
fi
cat missing-options.log
python -m pytest "$WORKSPACE/tests/cli" "$WORKSPACE/tests/contract"
)


## Record your findings

**Expected outcome:** All four command helps match their CWL inputs, required options are enforced, and CLI/contract tests pass.

Add your observations to a Markdown cell: what you predicted, what the command produced, and which contract declaration explains it. Restart the kernel and run all cells with the default settings to check reproducibility.

Compare [the solution](../modules/02-derive-the-cli/solution.md) after the activity. Return to [the lab index](README.md).
